# Lab 03: RAG and ANN

Build the two halves of retrieval by hand: a minimal RAG pipeline (retrieve, ground, cite, abstain) and an IVF approximate-nearest-neighbor index with its recall/scan tradeoff. Fill in the `TODO` cell; reference in `solution/`. Concepts: [rag-end-to-end](../../concepts/rag/rag-end-to-end.md), [similarity-and-ann](../../concepts/vector-db/similarity-and-ann.md).

## Step 0: retrieve, ground, cite

In [ ]:
from rag import Retriever, CORPUS
# RAG answers from retrieved context, not parameters. Chunk -> score -> ground -> cite.
r = Retriever()
out = r.answer("graph index for nearest neighbor search")
print("answer:", out["answer"])
print("cited source:", out["source"], " score:", round(out["score"], 2))

## Step 1: abstain when nothing is relevant

In [ ]:
# The behavior that separates RAG from a search box: when nothing is relevant, abstain instead of
# inventing an answer. An out-of-corpus query scores ~0 and is refused.
print(r.answer("what is the capital of France"))

## Step 2: exact vs. approximate (IVF) search

In [ ]:
from ann import make_vectors, build_ivf, exact_topk, ivf_search, N, NLIST
# Exact search compares the query against every vector - correct but linear. An IVF index clusters the
# vectors and searches only the nearest clusters.
pts = make_vectors(N, 16, seed=0)
cent, assign = build_ivf(pts, NLIST, seed=0)
q = make_vectors(1, 16, seed=7)[0]
truth = exact_topk(q, pts)
approx, scanned = ivf_search(q, pts, cent, assign, nprobe=2)
print("exact top-5:   ", truth)
print("ivf top-5:     ", approx, f"(scanned {scanned}/{N})")

## Step 3: the recall vs. scan tradeoff

In [ ]:
# TODO: probe the IVF index at several nprobe values and tabulate recall@5 vs. vectors scanned.
# Why does recall rise as nprobe rises, and what does nprobe = nlist reduce to? (Use evaluate().)
raise NotImplementedError

## What you built

The two halves of retrieval, from scratch. A **RAG pipeline** that chunks a corpus, scores chunks against a query with TF-IDF cosine, and answers from the best chunk with a citation - or abstains when nothing is relevant, the behavior that keeps RAG from hallucinating. And an **ANN index** (IVF) that makes retrieval scale: cluster the vectors, search only the nearest clusters, and ride the recall-vs-scan curve - at one cluster, ~0.79 recall scanning 13% of the data; at all clusters, exact.

**Where this simplifies:** the retriever is lexical (TF-IDF); production adds dense embeddings and often hybrid search, and the generator is a real language model writing from the retrieved context rather than returning the chunk. The IVF index here is a small k-means; production indexes (HNSW, IVF+PQ) add graph navigation and compression - but the recall/latency/memory tradeoff you measured is exactly the one they tune. Concepts: [rag-end-to-end](../../concepts/rag/rag-end-to-end.md), [similarity-and-ann](../../concepts/vector-db/similarity-and-ann.md), [hnsw](../../concepts/vector-db/hnsw.md).